# PTS-JEPA — MUMDMC2025 E1 benchmark

Browser/GPU experiment for the first controlled comparison:

- ResNet-18 — supervised CNN baseline
- ViT-Tiny — supervised transformer baseline
- I-JEPA-Tiny — self-supervised representation learning + linear probe

The notebook downloads the public 2,500-image XPL subset directly from Figshare, audits the archive, creates a group-aware split from the MUMDMC filename convention, runs the three models, and writes results to `outputs/e1/`.

In [ ]:
# Kaggle notebook setup
import os, sys, subprocess, pathlib, json, re, zipfile, shutil
ROOT = pathlib.Path('/kaggle/working/pts-jepa')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/carbonvalanceis4-art/pts-jepa.git', str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repo:', ROOT)

In [ ]:
import torch, torchvision, timm
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('timm', timm.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
# Download and prepare the public subset.
subprocess.run([sys.executable, 'data/prepare_mumdmc.py', '--download'], check=True)
print('prepared')

In [ ]:
# Audit the prepared manifest before training.
import csv, collections, pandas as pd
manifest_path = pathlib.Path('data/manifests/mumdmc2025.csv')
df = pd.read_csv(manifest_path)
print('images:', len(df))
print('labels:
', df['label'].value_counts(dropna=False))
print('groups:', df['group_id'].nunique())
print(df.head())
assert len(df) == 2500, f'Expected 2500 images, found {len(df)}'
assert df['label'].notna().all() and (df['label'].astype(str).str.len() > 0).all(), 'Missing labels'

## Research split

MUMDMC filenames encode slide, mineral, photomicrograph type, crystal number, photo number, and rotation angle. We derive a specimen/crystal grouping key from the filename rather than randomly splitting images. The public subset is XPL-only, but this still protects us against duplicate/near-duplicate views if they occur in the cropped release.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
def derive_group(path):
    stem = pathlib.Path(path).stem
    parts = stem.split('-')
    # Expected style: slide-mineral-photomicrograph-crystal-photo-angle.
    # Group by slide + mineral + photomicrograph/crystal identifiers, excluding angle.
    if len(parts) >= 5:
        return '-'.join(parts[:5])
    return stem
df['research_group'] = df['path'].map(derive_group)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(gss1.split(df, groups=df['research_group']))
trainval = df.iloc[trainval_idx].copy()
test = df.iloc[test_idx].copy()
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1764705882, random_state=43)
tr_idx, va_idx = next(gss2.split(trainval, groups=trainval['research_group']))
train = trainval.iloc[tr_idx].copy()
val = trainval.iloc[va_idx].copy()
print(len(train), len(val), len(test))
assert not set(train.research_group) & set(val.research_group)
assert not set(train.research_group) & set(test.research_group)
assert not set(val.research_group) & set(test.research_group)
print('split leakage checks passed')

In [ ]:
# Common image loader
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = sorted(df['label'].unique().tolist())
label_to_id = {x:i for i,x in enumerate(classes)}
DATA_ROOT = pathlib.Path('data/raw/mumdmc2025')
tfm_train = transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tfm_eval = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
class MUMD(Dataset):
    def __init__(self, frame, train=False): self.frame=frame.reset_index(drop=True); self.tfm=tfm_train if train else tfm_eval
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]; p=DATA_ROOT / r['path']; img=Image.open(p).convert('RGB'); return self.tfm(img), label_to_id[r['label']]
train_loader=DataLoader(MUMD(train,True), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader=DataLoader(MUMD(val), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader=DataLoader(MUMD(test), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
from models.baselines import make_resnet18, make_vit_tiny
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import time, copy
def run_supervised(model, name, epochs=5):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05); loss_fn=torch.nn.CrossEntropyLoss()
    best=None; best_f1=-1
    for ep in range(1,epochs+1):
        model.train()
        for x,y in train_loader:
            x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); loss=loss_fn(model(x),y); loss.backward(); opt.step()
        model.eval(); ys=[]; ps=[]
        with torch.no_grad():
            for x,y in val_loader:
                p=model(x.to(DEVICE)).argmax(1).cpu().numpy(); ys.extend(y.numpy()); ps.extend(p)
        f1=f1_score(ys,ps,average='macro'); print(name, ep, 'val_acc', accuracy_score(ys,ps), 'val_f1', f1)
        if f1>best_f1: best_f1=f1; best=copy.deepcopy(model.state_dict())
    model.load_state_dict(best); model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in test_loader:
            p=model(x.to(DEVICE)).argmax(1).cpu().numpy(); ys.extend(y.numpy()); ps.extend(p)
    return {'model':name,'accuracy':accuracy_score(ys,ps),'macro_f1':f1_score(ys,ps,average='macro'),'confusion_matrix':confusion_matrix(ys,ps).tolist()}
results=[]
results.append(run_supervised(make_resnet18(len(classes), pretrained=True), 'resnet18'))
results.append(run_supervised(make_vit_tiny(len(classes), pretrained=True), 'vit_tiny'))
results

## I-JEPA-Tiny

The repository model is intentionally inexpensive and is not a reproduction of Meta's large-scale I-JEPA training regime. We pretrain the encoder without mineral labels, then freeze it and train a linear classifier on the labeled training split.

In [ ]:
from models.ijepa_tiny import IJEPATiny, random_target_mask
def pretrain_ijepa(epochs=10):
    model=IJEPATiny(image_size=224, patch_size=16, dim=192, depth=4, heads=4).to(DEVICE)
    opt=torch.optim.AdamW(model.context_encoder.parameters(),lr=1e-3,weight_decay=0.05)
    for ep in range(1,epochs+1):
        model.train(); losses=[]
        for x,_ in train_loader:
            x=x.to(DEVICE,non_blocking=True); mask=random_target_mask(x.size(0),model.context_encoder.num_patches,device=x.device)
            opt.zero_grad(set_to_none=True); loss=model(x,mask); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); model.update_target() ; losses.append(loss.item())
        print('ijepa epoch',ep,'loss',float(np.mean(losses)))
    return model
ijepa=pretrain_ijepa(epochs=10)

In [ ]:
class FrozenJEPA(torch.nn.Module):
    def __init__(self, encoder, n_classes):
        super().__init__(); self.encoder=encoder; self.head=torch.nn.Linear(encoder.dim,n_classes)
        for p in self.encoder.parameters(): p.requires_grad=False
    def forward(self,x): return self.head(self.encoder(x))
jepa_clf=FrozenJEPA(ijepa.context_encoder,len(classes)).to(DEVICE)
opt=torch.optim.AdamW(jepa_clf.head.parameters(),lr=1e-2,weight_decay=1e-4)
for ep in range(20):
    jepa_clf.train()
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(); loss=torch.nn.functional.cross_entropy(jepa_clf(x),y); loss.backward(); opt.step()
jepa_clf.eval(); ys=[]; ps=[]
with torch.no_grad():
    for x,y in test_loader:
        ps.extend(jepa_clf(x.to(DEVICE)).argmax(1).cpu().numpy()); ys.extend(y.numpy())
results.append({'model':'ijepa_tiny','accuracy':accuracy_score(ys,ps),'macro_f1':f1_score(ys,ps,average='macro'),'confusion_matrix':confusion_matrix(ys,ps).tolist()})
results

In [ ]:
out=pathlib.Path('outputs/e1'); out.mkdir(parents=True,exist_ok=True)
pd.DataFrame([{k:v for k,v in r.items() if k!='confusion_matrix'} for r in results]).to_csv(out/'metrics.csv',index=False)
(out/'results.json').write_text(json.dumps(results,indent=2))
pd.DataFrame([{k:v for k,v in r.items() if k!='confusion_matrix'} for r in results])